In [ ]:

import os, sys, glob, shutil, time, json
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_human.npy", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(base+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.hybrid import product_disjoint_pair_masks
from src.metrics import macro_pr_auc
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

Xh=np.load(prev+"/features_human.npy"); Xl=np.load(prev+"/features_llm.npy")
E1=np.load(prev+"/features_eval_lex.npy"); E2=np.load(prev+"/features_eval_mixed.npy")
items=pd.read_parquet(base+"/items_human.parquet",columns=["id","category"])
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
ev1=pd.read_parquet(base+"/eval_pairs.parquet"); ev2=pd.read_parquet(base+"/eval_pairs_mixed.parquet")
lp=pd.read_parquet(base+"/llm_pairs_sel.parquet")
# Категорию LLM-пар надо брать из их собственного файла товаров: в `items_human` этих
# карточек нет, и `fillna` свалила бы все LLM-пары в одну безымянную категорию.
li=pd.read_parquet(base+"/llm_items_sel.parquet",columns=["id","category"])
cat_of=dict(zip(items["id"],items["category"].astype(str)))
cat_all=dict(cat_of); cat_all.update(zip(li["id"],li["category"].astype(str)))
cp=hm["id1"].map(cat_of).astype(str).to_numpy()
cl=lp["id1"].map(cat_all).fillna("?").astype(str).to_numpy()
print("LLM-пар без категории:", int((cl=="?").sum()), flush=True)
y=hm["target"].to_numpy(np.int8); yl=lp["label"].to_numpy(np.int8)
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
va=np.flatnonzero(vm); rel=np.flatnonzero(~vm)
c1=ev1["category"].astype(str).to_numpy(); c2=ev2["category"].astype(str).to_numpy()
y1=ev1["target"].to_numpy(np.int8); y2=ev2["target"].to_numpy(np.int8)
def macro(p,c,yy): return float(np.mean([average_precision_score(yy[c==k],p[c==k])
    for k in np.unique(c) if len(np.unique(yy[c==k]))>1]))
TUNED=dict(max_iter=1500,learning_rate=0.04,max_leaf_nodes=255,l2_regularization=2.0,
           random_state=0,early_stopping=False)
OLD=dict(max_iter=800,learning_rate=0.05,max_leaf_nodes=63,random_state=0,early_stopping=False)

X=np.vstack([Xl,Xh[rel]]); yy=np.concatenate([yl,y[rel]]); cc=np.concatenate([cl,cp[rel]])
log(f"обучающих {len(X):,}, категорий {len(set(cc))}")

def evaluate(tag,ph,p1,p2):
    np.save(f"/kaggle/working/p_{tag}_lex.npy",p1); np.save(f"/kaggle/working/p_{tag}_mix.npy",p2)
    log(f"{tag:<22} holdout {macro_pr_auc(y[va],ph,cp[va])[0]:.6f}  лексич {macro(p1,c1,y1):.6f}  смеш {macro(p2,c2,y2):.6f}")

g_old=HistGradientBoostingClassifier(**OLD).fit(X,yy)
evaluate("общая_старые",g_old.predict_proba(Xh[va])[:,1],g_old.predict_proba(E1)[:,1],g_old.predict_proba(E2)[:,1])
g=HistGradientBoostingClassifier(**TUNED).fit(X,yy)
pg_h,pg_1,pg_2=g.predict_proba(Xh[va])[:,1],g.predict_proba(E1)[:,1],g.predict_proba(E2)[:,1]
evaluate("общая_подобранные",pg_h,pg_1,pg_2)

ph=pg_h.copy(); p1=pg_1.copy(); p2=pg_2.copy()
for k in sorted(set(cc)):
    idx=np.flatnonzero(cc==k)
    if len(idx)<3000 or len(np.unique(yy[idx]))<2: continue
    P=dict(TUNED); P["max_iter"]=800
    c=HistGradientBoostingClassifier(**P).fit(X[idx],yy[idx])
    for pred,mask,mat in ((ph,cp[va]==k,Xh[va]),(p1,c1==k,E1),(p2,c2==k,E2)):
        rows=np.flatnonzero(mask)
        if len(rows): pred[rows]=c.predict_proba(mat[rows])[:,1]
    log(f"  категорийная модель: {k} ({len(idx):,} пар)")
evaluate("по_категориям",ph,p1,p2)
for w in (0.3,0.5,0.7):
    evaluate(f"смесь_общей_и_кат_{w}", (1-w)*pg_h+w*ph, (1-w)*pg_1+w*p1, (1-w)*pg_2+w*p2)
log("готово")
